# Notebook 13 — Foundation Model Embeddings in Python
Goal: Learn the practical idea behind foundation model embeddings using a small, guided workflow.

This notebook focuses on the pattern:

`raw data → pretrained model → embedding vector → similarity / downstream task`

It uses a simple text example so the core embedding idea is easy to understand before moving to biological or multimodal examples.

## Section 1 — Import Libraries

In [ ]:
# If needed:
# pip install transformers torch scikit-learn pandas matplotlib

import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## Section 2 — What Is an Embedding?
An embedding is a numeric vector that represents an input in a learned feature space. The model learns to place semantically similar inputs close together in this vector space and dissimilar inputs far apart.

Key properties:
- similar inputs should have similar embeddings (high cosine similarity)
- different inputs should have different embeddings (low cosine similarity)
- vectors usually have hundreds or thousands of dimensions

Think of embeddings as compact distributed representations where each dimension captures some latent pattern (e.g., biology, disease, drug response) that the model has learned.

In practice, many foundation-model workflows are:
1. load a pretrained model
2. generate embeddings
3. compare or analyze those embeddings
4. optionally use them for clustering, retrieval, or downstream models

## Section 3 — Create a Small Example Dataset

In [ ]:
texts = [
    "KRAS mutation in colorectal cancer",
    "BRAF mutation in melanoma",
    "EGFR inhibitor response in lung cancer",
    "TP53 loss in tumor progression",
    "MAPK signaling pathway activation",
    "drug response in patient derived organoids"
]

df = pd.DataFrame({"text": texts})
df

## Section 4 — Load a Small Pretrained Model

We choose a lightweight model (`distilbert-base-uncased`) for fast experimentation. DistilBERT is a distilled version of BERT: it is smaller, faster, and still retains most of BERT’s semantic understanding, making it ideal for demonstration and prototyping at lower cost.

In real-world workflows, you may opt for a model with domain-specific training (e.g., BioBERT / PubMedBERT for biology) or a dedicated embedding model (e.g., `sentence-transformers`).

Loading the model includes tokenizer setup (text → tokens) and transformer model initialization (tokens → contextual embeddings).

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model_name

## Section 5 — Tokenize Text
Tokenization converts text into the numerical format expected by the model.

In [ ]:
example = texts[0]

inputs = tokenizer(example, return_tensors="pt", truncation=True, padding=True)
inputs

## Section 6 — Generate an Embedding for One Example
A common simple strategy is to average the final hidden states across tokens.

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
last_hidden_state.shape

In [ ]:
embedding = last_hidden_state.mean(dim=1).squeeze().numpy()
embedding.shape

## Section 7 — Wrap Embedding Extraction in a Function

This helper function keeps the workflow reusable and testable. It performs tokenization, runs a forward pass through the model, and pools token-level hidden states into a single vector.

We use mean pooling as a simple aggregation strategy. Other options include CLS token output, max pooling, or domain-specific pooling for better semantic quality.

In [ ]:
def get_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

emb = get_embedding("KRAS mutation in colorectal cancer", tokenizer, model)
emb.shape

## Section 8 — Embed All Examples

In [ ]:
embeddings = [get_embedding(text, tokenizer, model) for text in texts]

len(embeddings), embeddings[0].shape

In [ ]:
embedding_df = pd.DataFrame(embeddings)
embedding_df.head()

## Section 9 — Compare Embeddings with Cosine Similarity
Cosine similarity is a standard way to measure how close two embeddings are. It is scale-invariant and focuses on direction in embedding space, making it a good default for comparing semantic relatedness.

Expected interpretation:
- values near 1.0: very similar meaning
- values near 0.0: unrelated meaning
- values near -1.0: opposite direction (rare with embeddings from the same model)

In [ ]:
similarity_matrix = cosine_similarity(embeddings)

sim_df = pd.DataFrame(similarity_matrix, index=texts, columns=texts)
sim_df.round(3)

## Section 10 — Reduce Embeddings with PCA
Embeddings are usually high-dimensional, which is hard to inspect directly. PCA reduces dimensionality while preserving the maximum variance, so we can visualize clusters and relative distances in 2D.

Important note: PCA on embeddings is for exploration only and does not affect similarity metrics used in production.

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

plot_df = pd.DataFrame(coords, columns=["PC1", "PC2"])
plot_df["text"] = texts
plot_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_df["PC1"], plot_df["PC2"])

for _, row in plot_df.iterrows():
    plt.text(row["PC1"], row["PC2"], row["text"], fontsize=8)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Foundation Model Embeddings")
plt.show()

## Section 11 — Simple Interpretation
Questions to ask:
- Which texts are close together?
- Which texts seem biologically or conceptually related?
- Do pathway-related texts cluster separately from mutation-related texts?

## Section 12 — Expected Intuition Summary
After running this notebook, you should have gained these instincts:
- Text similarity is captured by vector direction in embedding space, not by raw token overlap.
- Cosine similarity provides a quick pairwise semantic distance signal.
- PCA reveals broad clusters but is not a production similarity metric.
- Embeddings are reusable representations; once computed, they support retrieval, clustering, and downstream classifiers with minimal extra work.

## Section 13 — Exercises

1. Add two more biological text examples and regenerate embeddings.
2. Identify the most similar pair of texts using the similarity matrix.
3. Change the wording of one text and see whether its embedding changes.
4. Compute similarity between two chosen texts directly.
5. Re-run the PCA plot after adding more examples.

## Skills Gained
- understanding what an embedding is
- loading a pretrained foundation model
- tokenizing inputs
- generating embedding vectors
- comparing embeddings with cosine similarity
- visualizing embeddings with PCA

This is the core pattern behind many practical foundation-model workflows.